# Feature Importance Analysis
AML Benchmark - Part A v2
Runtime: ~2 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
result = subprocess.run(
    ['git', 'clone', 'https://github.com/fdrmic/classimbalance', '/content/classimbalance'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [ ]:
%cd /content/classimbalance
!git checkout feature/account-level-features
!pip install -e . --quiet

In [ ]:
from pathlib import Path

RUNS_DIR = Path('/content/drive/MyDrive/aml_results/large_run_v2_ongoing/runs')

run_dirs = [d for d in sorted(RUNS_DIR.iterdir()) if d.is_dir()]
print(f'Total run folders found: {len(run_dirs)}')

complete = []
for d in run_dirs:
    if (d / 'model.pkl').exists() and (d / 'run_config.json').exists():
        complete.append(d)

print(f'Complete runs: {len(complete)}')
for d in complete:
    print(f'  {d.name}')

In [ ]:
import json
import joblib
import pandas as pd

records = []

for run_dir in complete:
    with open(run_dir / 'run_config.json') as f:
        cfg = json.load(f)

    model = joblib.load(run_dir / 'model.pkl')

    if not hasattr(model, 'feature_importances_'):
        print(f'Skipping {run_dir.name}: no feature_importances_')
        continue

    importances = model.feature_importances_
    feature_names = cfg.get('feature_names', [])

    if len(importances) != len(feature_names):
        print(f'Skipping {run_dir.name}: shape mismatch')
        continue

    for feat, imp in zip(feature_names, importances):
        records.append({
            'run_id': cfg['run_id'],
            'model': cfg['model'],
            'strategy': cfg['strategy'],
            'target_prevalence': cfg['target_prevalence'],
            'feature': feat,
            'importance': imp,
        })

df = pd.DataFrame(records)
print(f'Total records: {len(df)}')
print(f'Runs processed: {df["run_id"].nunique()}')

In [ ]:
xgb_mean = (
    df[df['model'] == 'xgboost']
    .groupby('feature')['importance']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
xgb_mean.columns = ['feature', 'mean_importance']

rf_mean = (
    df[df['model'] == 'random_forest']
    .groupby('feature')['importance']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
rf_mean.columns = ['feature', 'mean_importance']

per_strategy = (
    df[df['model'] == 'xgboost']
    .groupby(['strategy', 'feature'])['importance']
    .mean()
    .reset_index()
    .sort_values(['strategy', 'importance'], ascending=[True, False])
)

print('=== TOP 15 FEATURES - XGBoost ===')
print(xgb_mean.head(15).to_string(index=False))
print('\n=== TOP 15 FEATURES - Random Forest ===')
print(rf_mean.head(15).to_string(index=False))

In [ ]:
print('=== TOP 5 PER STRATEGY - XGBoost ===')
for strategy in sorted(per_strategy['strategy'].unique()):
    top5 = per_strategy[per_strategy['strategy'] == strategy].head(5)
    print(f'\n{strategy}:')
    print(top5[['feature', 'importance']].to_string(index=False))

In [ ]:
OUT_DIR = Path('/content/drive/MyDrive/aml_results/large_run_v2_ongoing/feature_importance')
OUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(OUT_DIR / 'feature_importance_all_runs.csv', index=False)
xgb_mean.to_csv(OUT_DIR / 'feature_importance_xgboost_mean.csv', index=False)
rf_mean.to_csv(OUT_DIR / 'feature_importance_rf_mean.csv', index=False)
per_strategy.to_csv(OUT_DIR / 'feature_importance_per_strategy.csv', index=False)

print(f'Saved to {OUT_DIR}')
for f in OUT_DIR.iterdir():
    print(f'  {f.name}')